# TESTING MODEL

## 1. Pytorch model

In [4]:
import torch
import torchaudio
import torch.nn.functional as F
from IPython.display import Audio
from tkinter.filedialog import askopenfilename
import tkinter as tk

# === Class labels ===
class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]

# === Load model ===
model = torch.load("cnn_model_full.pth", weights_only=False)
model.eval()

# === Auto-detect expected CNN input shape ===
final_shape = None
for h in range(30, 200):  # height
    for w in range(30, 200):  # width
        try:
            dummy = torch.zeros((1, 1, h, w))
            _ = model(dummy)
            final_shape = (h, w)
            break
        except Exception:
            continue
    if final_shape:
        break

if final_shape is None:
    raise RuntimeError("❌ Could not determine model's expected input shape automatically.")
else:
    print(f"✅ CNN expects input shape: {final_shape}")

# === Predict Function ===
def predict_new_audio():
    # File picker
    tk.Tk().withdraw()
    filename = askopenfilename(filetypes=[("WAV files", "*.wav")])
    if not filename:
        print("❌ No file selected.")
        return

    waveform, sr = torchaudio.load(filename)
    waveform = waveform.mean(dim=0, keepdim=True)

    if sr != 22050:
        waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=22050)(waveform)

    waveform = torchaudio.transforms.Vad(sample_rate=22050)(waveform)

    # === Extract MFCC + delta + delta-delta
    mfcc = torchaudio.transforms.MFCC(
        sample_rate=22050, n_mfcc=13,
        melkwargs={"n_fft": 1024, "hop_length": 512, "n_mels": 40}
    )(waveform)
    delta = torchaudio.functional.compute_deltas(mfcc)
    delta2 = torchaudio.functional.compute_deltas(delta)
    features = torch.cat((mfcc, delta, delta2), dim=1)  # shape: [1, 39, T]

    h, w = final_shape

    # Pad/crop height (dim=1)
    if features.shape[1] < h:
        features = F.pad(features, (0, 0, 0, h - features.shape[1]))
    else:
        features = features[:, :h, :]

    # Pad/crop width (dim=2)
    if features.shape[2] < w:
        features = F.pad(features, (0, w - features.shape[2]))
    else:
        features = features[:, :, :w]

    # Final shape: [1, 1, H, W]
    features = features.unsqueeze(0).to(torch.float32)

    print("📦 Input shape:", features.shape)

    with torch.no_grad():
        output = model(features)
        predicted = output.argmax(dim=1).item()

    print(f"🍼 Predicted Cry Type: {class_names[predicted]}")
    return Audio(filename)

# === Run it
predict_new_audio()


✅ CNN expects input shape: (128, 184)
📦 Input shape: torch.Size([1, 1, 128, 184])
🍼 Predicted Cry Type: tired


## 2. Keras model

In [2]:
import numpy as np
import librosa
import tensorflow as tf
from keras.models import load_model
from keras.layers import Layer
from tkinter.filedialog import askopenfilename
import tkinter as tk
from IPython.display import Audio

# === Class labels ===
class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]

# === Define the custom Attention Layer ===
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='normal', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(inputs, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = inputs * a
        return tf.keras.backend.sum(output, axis=1)

# === Load the model with AttentionLayer ===
model = load_model("best_model.keras", custom_objects={'AttentionLayer': AttentionLayer}, compile=False)
print("✅ Model loaded.")

# === Get expected input shape (e.g., 187 x 130 x 1) ===
input_shape = model.input_shape  # (None, 187, 130, 1)
H, W = input_shape[1], input_shape[2]
print(f"✅ Model expects input shape: ({H}, {W})")

# === Feature extraction using MFCC + delta + delta2 ===
def extract_features(filepath, sr=22050, expected_shape=(187, 130)):
    y, _ = librosa.load(filepath, sr=sr)
    y = librosa.util.normalize(y)
    y, _ = librosa.effects.trim(y)

    if len(y) < sr:
        y = np.pad(y, (0, sr - len(y)))
    else:
        y = y[:sr]

    # Use 63 MFCCs to create 189 total features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=63)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    features = np.concatenate([mfcc, delta, delta2], axis=0)  # shape: (189, T)

    # Slice to exactly 187 features
    features = features[:187, :]

    # Pad/crop time axis to match model width
    if features.shape[1] < expected_shape[1]:
        pad_width = expected_shape[1] - features.shape[1]
        features = np.pad(features, ((0, 0), (0, pad_width)), mode='constant')
    else:
        features = features[:, :expected_shape[1]]

    return features

# === Predict a new audio file ===
def predict_new_audio():
    tk.Tk().withdraw()
    filepath = askopenfilename(filetypes=[("WAV files", "*.wav")])
    if not filepath:
        print("❌ No file selected.")
        return

    print(f"📂 Selected file: {filepath}")
    features = extract_features(filepath, expected_shape=(H, W))  # shape: (187, 130)
    input_tensor = np.expand_dims(features, axis=(0, -1))  # shape: (1, 187, 130, 1)

    prediction = model.predict(input_tensor)
    predicted_class = np.argmax(prediction)

    print(f"🍼 Predicted Cry Type: {class_names[predicted_class]}")
    return Audio(filepath)

# === Run the prediction ===
predict_new_audio()


✅ Model loaded.
✅ Model expects input shape: (187, 130)
📂 Selected file: C:/Users/user/Desktop/DATASET SEMUA/infant cry dataset/Dataset/wav format/cry/186c.wav
1/1 [==============================] - 1s 1s/step
🍼 Predicted Cry Type: tired


In [16]:
import numpy as np
import librosa
import tensorflow as tf
from keras.models import load_model
from keras.layers import Layer
from tkinter.filedialog import askopenfilename
import tkinter as tk
from IPython.display import Audio

# === Class labels ===
class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]

# === Define the custom Attention Layer ===
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='normal', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(inputs, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = inputs * a
        return tf.keras.backend.sum(output, axis=1)

# === Load the model with AttentionLayer ===
model = load_model("best_model_ResLSTM.h5", custom_objects={'AttentionLayer': AttentionLayer}, compile=False)
print("✅ Model loaded.")

# === Get expected input shape (e.g., 187 x 130 x 1) ===
input_shape = model.input_shape  # (None, 187, 130, 1)
H, W = input_shape[1], input_shape[2]
print(f"✅ Model expects input shape: ({H}, {W})")

# === Feature extraction using MFCC + delta + delta2 ===
def extract_features(filepath, sr=22050, expected_shape=(187, 130)):
    y, _ = librosa.load(filepath, sr=sr)
    y = librosa.util.normalize(y)
    y, _ = librosa.effects.trim(y)

    if len(y) < sr:
        y = np.pad(y, (0, sr - len(y)))
    else:
        y = y[:sr]

    # Use 63 MFCCs to create 189 total features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=63)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    features = np.concatenate([mfcc, delta, delta2], axis=0)  # shape: (189, T)

    # Slice to exactly 187 features
    features = features[:187, :]

    # Pad/crop time axis to match model width
    if features.shape[1] < expected_shape[1]:
        pad_width = expected_shape[1] - features.shape[1]
        features = np.pad(features, ((0, 0), (0, pad_width)), mode='constant')
    else:
        features = features[:, :expected_shape[1]]

    return features

# === Predict a new audio file ===
def predict_new_audio():
    tk.Tk().withdraw()
    filepath = askopenfilename(filetypes=[("WAV files", "*.wav")])
    if not filepath:
        print("❌ No file selected.")
        return

    print(f"📂 Selected file: {filepath}")
    features = extract_features(filepath, expected_shape=(H, W))  # shape: (187, 130)
    input_tensor = np.expand_dims(features, axis=(0, -1))  # shape: (1, 187, 130, 1)

    prediction = model.predict(input_tensor)
    predicted_class = np.argmax(prediction)

    print(f"🍼 Predicted Cry Type: {class_names[predicted_class]}")
    return Audio(filepath)

# === Run the prediction ===
predict_new_audio()


✅ Model loaded.
✅ Model expects input shape: (186, 130)
📂 Selected file: C:/Users/user/Desktop/DATASET SEMUA/Baby Cry Pattern Archive/cry/hungry/9aa8bac5-eeb9-4f19-a4bf-7c439e87364b-1430745385374-1.7-m-04-hu.wav


ValueError: in user code:

    File "C:\Users\user\anaconda3\Lib\site-packages\keras\engine\training.py", line 2169, in predict_function  *
        return step_function(self, iterator)
    File "C:\Users\user\anaconda3\Lib\site-packages\keras\engine\training.py", line 2155, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\user\anaconda3\Lib\site-packages\keras\engine\training.py", line 2143, in run_step  **
        outputs = model.predict_step(data)
    File "C:\Users\user\anaconda3\Lib\site-packages\keras\engine\training.py", line 2111, in predict_step
        return self(x, training=False)
    File "C:\Users\user\anaconda3\Lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\user\anaconda3\Lib\site-packages\keras\engine\input_spec.py", line 298, in assert_input_compatibility
        raise ValueError(

    ValueError: Input 0 o